# Feature 1 — HMM 소음 대비 신호 국면 판별기 v2

Shiller/FRED/yfinance 데이터 → 8개 이성/감정 지표 → HMM(4상태) → 국면 판별

| rank | 국면 | 의미 |
|------|------|------|
| 0 | 펀더멘털 반영 | 주가가 실적·데이터를 충실히 반영 |
| 1 | 펀더멘털 약반영 | 대체로 논리적, 센티멘트 일부 혼재 |
| 2 | 센티멘트 약반영 | 심리·감정이 데이터보다 앞서는 구간 |
| 3 | 센티멘트 지배 | 펀더멘털과 무관하게 움직이는 구간 |

## Q1. 환경 설정 + 상수 정의
- 필요한 라이브러리를 임포트하고 HMM 모델의 상수를 정의하세요
- `GaussianHMM`, `RobustScaler` 클래스명과 `N_STATES`, 피처 이름을 채우세요

In [ ]:
# ── Cell 1-A: 라이브러리 임포트 ──────────────────────────────────────────
import sys, warnings  # 시스템 모듈과 경고 필터 임포트
sys.path.insert(0, '..')  # 상위 디렉토리를 경로에 추가
warnings.filterwarnings('ignore')  # 경고 메시지 무시

import datetime  # 날짜/시간 처리 모듈
import numpy as np  # 수치 계산 라이브러리
import pandas as pd  # 데이터프레임 처리 라이브러리
import matplotlib.pyplot as plt  # 시각화 라이브러리
import matplotlib.font_manager as fm  # 폰트 매니저
import seaborn as sns  # 통계 시각화 라이브러리
import yfinance as yf  # 야후 파이낸스 데이터 수집
import requests, time  # HTTP 요청 및 시간 지연 모듈
from io import StringIO  # 문자열을 파일처럼 읽기 위한 모듈

from hmmlearn.hmm import ___  # HMM 가우시안 모델 클래스 (Q1-1)
from sklearn.preprocessing import ___  # 로버스트 스케일러 클래스 (Q1-2)

from dotenv import load_dotenv  # 환경변수 로드 모듈
load_dotenv('../.env')  # .env 파일에서 환경변수 로드

# 한글 폰트 설정 (Windows 환경)
font_path = 'C:/Windows/Fonts/malgun.ttf'  # 맑은 고딕 폰트 경로
font_name = fm.FontProperties(fname=font_path).get_name()  # 폰트 이름 추출
plt.rc('font', family=font_name)  # matplotlib 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

print('환경 설정 완료')  # 설정 완료 메시지 출력

In [ ]:
# ── Cell 1-B: 상수 정의 ──────────────────────────────────────────────────
N_STATES = ___  # HMM 상태 수: 4개 국면 (Q1-3)
PHASE_NAMES  = {0: '🧠 펀더멘털 반영', 1: '⚖️ 펀더멘털 약반영', 2: '🌊 센티멘트 약반영', 3: '🔥 센티멘트 지배'}  # 국면 이름 딕셔너리
PHASE_COLORS = {0: '#4CAF50', 1: '#8BC34A', 2: '#FF9800', 3: '#F44336'}  # 국면별 색상 딕셔너리
RANK_TO_PHASE = [0, 1, 2, 3]  # noise_score 오름차순 순위 → 국면 매핑

FEATURE_NAMES = ['___', '___', '___',  # 피처명 리스트: fundamental_gap, erp_zscore, residual_corr (Q1-4)
                 '___', '___', '___', '___',  # dispersion, amihud, vix_term, hy_spread (Q1-5)
                 '___']  # realized_vol (Q1-6)

HEADERS = {'User-Agent': 'Mozilla/5.0'}  # HTTP 요청 헤더 (봇 차단 우회)

print(f'국면: {list(PHASE_NAMES.values())}')  # 국면 이름 출력
print(f'피처: {FEATURE_NAMES}')  # 피처 이름 출력

## Q2. Shiller 월별 데이터 수집 (P, E, CAPE)
- Shiller ie_data.xls에서 S&P 500 Price, Earnings, CAPE를 수집합니다
- 날짜 파싱, Yahoo Finance 보완, CAPE 계산 로직을 채우세요

In [ ]:
# ── Cell 2-A: Shiller 데이터 다운로드 + CAPE 컬럼 탐색 ────────────────────
SHILLER_URL = 'http://www.econ.yale.edu/~shiller/data/ie_data.xls'  # Shiller 데이터 URL

print('Shiller ie_data.xls 다운로드 중...')  # 다운로드 상태 메시지
shiller = pd.read_excel(SHILLER_URL, sheet_name='Data', skiprows=7, header=0)  # 엑셀 파일 읽기 (7행 건너뜀)

cols = shiller.columns.tolist()  # 컬럼명 리스트 추출
shiller = shiller.rename(columns={cols[0]: 'date_raw', cols[1]: 'P', cols[3]: 'E'})  # 주요 컬럼명 변경

# CAPE 컬럼 자동 탐색
cape_col = None  # CAPE 컬럼명 초기화
for c in cols:  # 모든 컬럼을 순회하며
    if isinstance(c, str) and 'cape' in c.lower():  # 'cape' 문자열 포함 여부 확인
        cape_col = c  # CAPE 컬럼 발견
        break  # 탐색 종료
    if isinstance(c, str) and 'cyclically' in c.lower():  # 'cyclically' 포함 여부 확인
        cape_col = c  # CAPE 컬럼 발견
        break  # 탐색 종료

if cape_col is None:  # CAPE 컬럼을 찾지 못한 경우
    print('  ⚠ CAPE 컬럼 자동감지 실패 → P/E10으로 직접 계산')  # 경고 메시지
    cape_available = False  # CAPE 사용 불가 플래그
else:  # CAPE 컬럼을 찾은 경우
    shiller = shiller.rename(columns={cape_col: 'CAPE'})  # CAPE로 컬럼명 변경
    cape_available = True  # CAPE 사용 가능 플래그
    print(f'  ✓ CAPE 컬럼: {cape_col}')  # 발견된 컬럼명 출력

In [ ]:
# ── Cell 2-B: 날짜 파싱 + 숫자 변환 ──────────────────────────────────────
def parse_shiller_date(val):  # Shiller 날짜 형식 파싱 함수
    try:  # 예외 처리 시작
        s = str(val)  # 문자열로 변환
        parts = s.split('.')  # 소수점으로 분리
        year = int(parts[0])  # 연도 추출
        month = int(parts[1]) if len(parts) > 1 and parts[1] else 1  # 월 추출 (없으면 1월)
        month = max(1, min(___, month))  # 월 범위 제한: 1~12 (Q2-1)
        return pd.Timestamp(year=year, month=month, day=1)  # 타임스탬프 생성
    except:  # 파싱 실패 시
        return pd.NaT  # 결측값 반환

shiller['date'] = shiller['date_raw'].apply(parse_shiller_date)  # 날짜 컬럼 생성
shiller = shiller.___(subset=['date'])  # 날짜 결측 행 제거 (Q2-2)
shiller = shiller.set_index('date')  # 날짜를 인덱스로 설정
shiller = shiller[~shiller.index.duplicated(keep='last')]  # 중복 인덱스 제거 (마지막 값 유지)
shiller = shiller.sort_index()  # 인덱스 기준 정렬

# 숫자형 변환
for col in ['P', 'E']:  # 가격과 이익 컬럼에 대해
    shiller[col] = pd.to_numeric(shiller[col], errors='coerce')  # 숫자로 변환 (실패시 NaN)
if cape_available:  # CAPE 컬럼이 존재하면
    shiller['CAPE'] = pd.to_numeric(shiller['CAPE'], errors='coerce')  # CAPE도 숫자로 변환

print(f'Shiller 데이터: {shiller.index[0].date()} ~ {shiller.index[-1].date()}')  # 기간 출력

In [ ]:
# ── Cell 2-C: Yahoo Finance S&P 500 보완 ─────────────────────────────────
print('S&P 500 Yahoo Finance 보완 데이터 수집 중...')  # 보완 수집 시작 메시지

def _fetch_yahoo(ticker, years=3):  # Yahoo Finance에서 월별 데이터 수집 함수
    today = datetime.date.today()  # 오늘 날짜 가져오기
    from_date = today - datetime.timedelta(days=365 * years)  # 시작일 계산
    epoch = datetime.datetime(1970, 1, 1)  # 유닉스 에포크 기준점
    from_ts = int((datetime.datetime.combine(from_date, datetime.time()) - epoch).total_seconds())  # 시작 타임스탬프
    to_ts   = int((datetime.datetime.combine(today,     datetime.time()) - epoch).total_seconds())  # 종료 타임스탬프
    url = f'https://query1.finance.yahoo.com/v8/finance/chart/{ticker}'  # Yahoo API URL
    params = {'interval': '1mo', 'period1': from_ts, 'period2': to_ts}  # 월별 간격 파라미터
    resp = requests.get(url, params=params, headers=HEADERS, timeout=15)  # HTTP GET 요청
    resp.raise_for_status()  # HTTP 에러 확인
    result = resp.json()['chart']['result'][0]  # JSON 결과 파싱
    timestamps = result['timestamp']  # 타임스탬프 추출
    closes = result['indicators']['adjclose'][0]['adjclose']  # 수정종가 추출
    index = pd.to_datetime(timestamps, unit='s').normalize()  # 타임스탬프를 날짜로 변환
    return pd.Series(closes, index=index, name='P')  # 시리즈로 반환

sp_yahoo = _fetch_yahoo('^GSPC')  # S&P 500 데이터 수집
sp_yahoo.index = sp_yahoo.index.to_period('M').to_timestamp()  # 월초로 정규화

# Shiller P가 NaN인 최신 구간을 Yahoo로 채움
last_shiller_p = shiller['P'].dropna().index[-1]  # Shiller P의 마지막 유효 날짜
new_months = sp_yahoo[sp_yahoo.index > last_shiller_p]  # 보완이 필요한 구간
for dt, price in new_months.items():  # 새로운 월별 데이터를 순회
    if dt not in shiller.index:  # 인덱스에 없으면
        shiller.loc[dt] = np.nan  # NaN 행 생성
    shiller.loc[dt, 'P'] = price  # 가격 채움
shiller = shiller.sort_index()  # 인덱스 정렬

# E: forward-fill (최신 몇 달은 미발표 → 직전 값으로 채움)
shiller['E'] = shiller['E'].___()  # 이익 데이터 앞으로 채우기 (Q2-3)

print(f'  보완 후: {shiller.index[0].date()} ~ {shiller.index[-1].date()} ({len(shiller)}개월)')  # 결과 출력

In [ ]:
# ── Cell 2-D: CAPE 계산 + 펀더멘털 괴리 변수 ① ──────────────────────────
# CAPE: 없는 구간은 P / E10 (10년 평균 E)으로 계산
e10 = shiller['E'].___(___, min_periods=60).___()  # 10년(120개월) 롤링 평균 (Q2-4, Q2-5)
if cape_available:  # CAPE가 이미 있으면
    shiller['CAPE'] = shiller['CAPE'].fillna(shiller['P'] / e10)  # 결측만 P/E10으로 보완
else:  # CAPE가 없으면
    shiller['CAPE'] = shiller['P'] / e10  # 직접 계산
    cape_available = True  # 사용 가능 플래그 설정

shiller = shiller[shiller['P'].notna() & shiller['E'].notna()]  # P, E 모두 있는 행만 유지

print(f'  Shiller 최종: {shiller.index[0].date()} ~ {shiller.index[-1].date()} ({len(shiller)}개월)')  # 최종 범위 출력

# ── 변수 ① 펀더멘털 괴리: Δlog(P)_12m − Δlog(E)_12m ──
shiller['log_P'] = ___(shiller['P'])  # 가격의 자연로그 (Q2-6)
shiller['log_E'] = ___(shiller['E'].___(lower=0.01))  # 이익의 자연로그 (하한 클리핑) (Q2-7, Q2-8)

fundamental_gap = shiller['log_P'].___(___)  - shiller['log_E'].___(___) # 12개월 로그 변화율의 차이 (Q2-9, Q2-10)
fundamental_gap = fundamental_gap.___()  # 결측값 제거 (Q2-11)
fundamental_gap.name = 'fundamental_gap'  # 시리즈 이름 설정

print(f'\n변수 ① 펀더멘털 괴리: {len(fundamental_gap)}개월')  # 관측 수 출력
print(f'  기간: {fundamental_gap.index[0].date()} ~ {fundamental_gap.index[-1].date()}')  # 기간 출력
print(f'  평균: {fundamental_gap.___():.4f}  |  표준편차: {fundamental_gap.___():.4f}')  # 기술통계 출력 (Q2-12, Q2-13)

In [ ]:
# ── Cell 2-E: 펀더멘털 괴리 시각화 ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)  # 2행 1열 서브플롯 생성

axes[0].plot(shiller.index, shiller['log_P'].diff(12), label='Δlog(P) 12m', color='#2196F3')  # 가격 로그변화율 플롯
axes[0].plot(shiller.index, shiller['log_E'].diff(12), label='Δlog(E) 12m', color='#FF9800')  # 이익 로그변화율 플롯
axes[0].legend()  # 범례 추가
axes[0].set_title('S&P 500 가격 vs 이익 12개월 로그변화율')  # 제목 설정
axes[0].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[0].grid(alpha=0.3)  # 격자 표시

axes[1].fill_between(fundamental_gap.index, fundamental_gap, 0,
                     where=fundamental_gap > 0, color='#F44336', alpha=0.4, label='주가 > 이익')  # 양수 영역 채움
axes[1].fill_between(fundamental_gap.index, fundamental_gap, 0,
                     where=fundamental_gap <= 0, color='#2196F3', alpha=0.4, label='주가 < 이익')  # 음수 영역 채움
axes[1].set_title('① 펀더멘털 괴리 = Δlog(P) − Δlog(E)')  # 제목 설정
axes[1].set_ylabel('괴리')  # Y축 라벨
axes[1].legend()  # 범례 추가
axes[1].grid(alpha=0.3)  # 격자 표시

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

## Q3. FRED 데이터 일괄 수집
- FRED에서 TIPS 실질금리, VIX, VIX3M, 하이일드 OAS를 수집합니다
- `resample`, `dropna` 등의 메서드를 채우세요

In [ ]:
# ── Cell 3: FRED 데이터 일괄 수집 ──────────────────────────────────────────
FRED_BASE = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='  # FRED CSV API 기본 URL

def fetch_fred(series_id, col_name, retries=4, timeout=30):  # FRED 데이터 수집 함수
    """FRED CSV 다운로드 (지수 백오프 재시도)."""  # 독스트링
    url = FRED_BASE + series_id  # 시리즈별 URL 생성
    for attempt in range(retries):  # 재시도 루프
        try:  # 예외 처리 시작
            resp = requests.get(url, headers=HEADERS, timeout=timeout)  # HTTP 요청
            resp.raise_for_status()  # 에러 확인
            df = pd.read_csv(StringIO(resp.text), index_col=0, parse_dates=True)  # CSV 파싱
            df.columns = [col_name]  # 컬럼명 설정
            df[col_name] = pd.to_numeric(df[col_name], errors='coerce')  # 숫자로 변환
            return df  # 데이터프레임 반환
        except Exception:  # 예외 발생 시
            if attempt < retries - 1:  # 마지막 시도가 아니면
                wait = 2 ** attempt  # 지수 백오프 대기시간
                print(f'  [{series_id}] 재시도 {attempt+1}/{retries} ({wait}초 대기)...')  # 재시도 메시지
                time.sleep(wait)  # 대기
            else:  # 마지막 시도에서도 실패하면
                raise  # 예외 발생

print('FRED 데이터 수집 중...')  # 수집 시작 메시지

# DFII10: 10년 TIPS 실질금리 (일별)
df_tips = fetch_fred('DFII10', 'tips_rate')  # TIPS 실질금리 수집
tips_monthly = df_tips['tips_rate'].resample('MS').last().___()  # 월별 마지막 값 + 결측 제거 (Q3-1)
print(f'  ✓ DFII10 (TIPS 실질금리): {len(tips_monthly)}개월')  # 수집 결과 출력

# VIXCLS: VIX 일별
df_vix = fetch_fred('VIXCLS', 'vix')  # VIX 수집
vix_monthly = df_vix['vix'].resample('MS').last().___()  # 월별 마지막 값 + 결측 제거 (Q3-2)
print(f'  ✓ VIXCLS (VIX): {len(vix_monthly)}개월')  # 수집 결과 출력

# VXVCLS: VIX3M 일별 (2007.12~)
df_vix3m = fetch_fred('VXVCLS', 'vix3m')  # VIX3M 수집
vix3m_monthly = df_vix3m['vix3m'].resample('MS').last().___()  # 월별 마지막 값 + 결측 제거 (Q3-3)
print(f'  ✓ VXVCLS (VIX3M): {len(vix3m_monthly)}개월')  # 수집 결과 출력

# BAMLH0A0HYM2: ICE BofA US High Yield OAS (일별)
df_hy = fetch_fred('BAMLH0A0HYM2', 'hy_spread')  # 하이일드 OAS 수집
hy_monthly = df_hy['hy_spread'].resample('MS').last().___()  # 월별 마지막 값 + 결측 제거 (Q3-4)
print(f'  ✓ BAMLH0A0HYM2 (HY OAS): {len(hy_monthly)}개월')  # 수집 결과 출력

print('\nFRED 수집 완료')  # 수집 완료 메시지

### 변수 ② 실질 ERP 괴리 (10년 Z-Score)
- 실질이익수익률 = 1/CAPE (Shiller CAPE 기반)
- 실질 ERP = 실질이익수익률 − TIPS 실질금리 (DFII10)
- 10년(120개월) 롤링 Z-Score → 절대값 = 양방향 괴리 측정
- 크면 → 밸류에이션이 역사적 범위에서 크게 이탈 (감정적)

## Q4. 변수 ② 실질 ERP Z-Score 계산
- 실질 ERP = 1/CAPE - TIPS 실질금리
- 120개월 롤링 Z-Score의 절대값을 구하세요

In [ ]:
# ── Cell 4-A: 변수 ② 실질 ERP Z-Score 계산 ──────────────────────────────
# Shiller CAPE에서 실질이익수익률 계산
if cape_available:  # CAPE 데이터가 있으면
    cape_series = shiller['CAPE'].___()  # CAPE 결측 제거 (Q4-1)
else:  # CAPE가 없으면 직접 계산
    e10 = shiller['E'].___(120, min_periods=60).___()  # 10년 롤링 평균 (Q4-2, Q4-3)
    cape_series = shiller['P'] / e10  # P/E10으로 CAPE 계산
    cape_series = cape_series.___()  # 결측 제거 (Q4-4)

real_earnings_yield = 1.0 / cape_series  # 실질이익수익률 = 1/CAPE

# TIPS 실질금리 (월별, % → 소수 변환)
tips_aligned = tips_monthly / 100.0  # 퍼센트를 소수로 변환

# 실질 ERP = 실질이익수익률 − TIPS 실질금리
erp_df = pd.DataFrame({  # ERP 계산용 데이터프레임 생성
    'earnings_yield': real_earnings_yield,  # 이익수익률 컬럼
    'tips_rate': tips_aligned,  # TIPS 금리 컬럼
}).___()  # 결측 행 제거 (Q4-5)
erp = erp_df['earnings_yield'] - erp_df['tips_rate']  # ERP 계산

# 10년(120개월) 롤링 Z-Score
erp_roll_mean = erp.___(___, min_periods=60).___()  # 120개월 롤링 평균 (Q4-6, Q4-7)
erp_roll_std  = erp.___(___, min_periods=60).___()  # 120개월 롤링 표준편차 (Q4-8, Q4-9)
erp_zscore = ((erp - erp_roll_mean) / erp_roll_std).___()  # Z-Score 절대값 (Q4-10)
erp_zscore = erp_zscore.___()  # 결측 제거 (Q4-11)
erp_zscore.name = 'erp_zscore'  # 시리즈 이름 설정

print(f'변수 ② 실질 ERP Z-Score: {len(erp_zscore)}개월')  # 관측 수 출력
print(f'  기간: {erp_zscore.index[0].date()} ~ {erp_zscore.index[-1].date()}')  # 기간 출력
print(f'  평균: {erp_zscore.___():.4f}  |  표준편차: {erp_zscore.___():.4f}')  # 기술통계 (Q4-12, Q4-13)

In [ ]:
# ── Cell 4-B: ERP Z-Score 시각화 ─────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)  # 2행 1열 서브플롯

axes[0].plot(erp.index, erp, label='실질 ERP (1/CAPE − TIPS)', color='#4CAF50')  # ERP 플롯
axes[0].plot(erp_roll_mean.index, erp_roll_mean, label='10년 롤링 평균', color='gray', linestyle='--')  # 롤링 평균
axes[0].legend()  # 범례
axes[0].set_title('실질 주식위험프리미엄 (ERP)')  # 제목
axes[0].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[0].grid(alpha=0.3)  # 격자

axes[1].fill_between(erp_zscore.index, erp_zscore, 0, color='#FF9800', alpha=0.5)  # Z-Score 영역 채움
axes[1].axhline(2, color='red', linestyle='--', alpha=0.5, label='Z=2 (2σ 이탈)')  # 2시그마 기준선
axes[1].set_title('② 실질 ERP Z-Score (절대값, 10년 롤링)')  # 제목
axes[1].set_ylabel('|Z-Score|')  # Y축 라벨
axes[1].legend()  # 범례
axes[1].grid(alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

### 변수 ③ 섹터 동조화 (베타 제거 잔차 상관 + 크로스섹션 디스퍼전)
- 5섹터 × 5종목의 일별 수익률에서 SPY 시장 베타 제거 → 잔차 수익률
- 잔차 20일 롤링 페어와이즈 상관 → 전체 평균 = `residual_corr`
- 전 종목 일별 수익률의 횡단면 표준편차 → 20일 롤링 = `dispersion`

## Q5. 섹터 동조화 - 베타 제거 + 잔차 상관 + 디스퍼전
- `.cov()`, `.var()`, `.rolling()`, `.corr()`, `.std()`, `.pct_change()` 등을 채우세요

In [ ]:
# ── Cell 5-A: 섹터 종목 데이터 수집 ──────────────────────────────────────
from itertools import combinations  # 조합 생성 모듈

SECTOR_STOCKS = {
    'XLK': ['AAPL', 'MSFT', 'NVDA', 'AVGO', 'CRM'],
    'XLF': ['JPM', 'BAC', 'WFC', 'GS', 'MS'],
    'XLE': ['XOM', 'CVX', 'COP', 'SLB', 'EOG'],
    'XLV': ['UNH', 'JNJ', 'LLY', 'PFE', 'ABT'],
    'XLI': ['CAT', 'HON', 'UNP', 'GE', 'RTX'],
}  # 5섹터 × 5종목 딕셔너리
all_stocks = [s for stocks in SECTOR_STOCKS.values() for s in stocks]  # 전체 종목 리스트 평탄화

print(f'{len(all_stocks)}개 종목 + SPY 일별 종가 수집 중...')  # 수집 시작 메시지
stock_frames = {}  # 종목별 데이터 저장 딕셔너리
start_date = str(datetime.date.today() - datetime.timedelta(days=365 * 18 + 30))  # 18년 전 시작일

# SPY도 함께 수집 (베타 제거용)
for ticker in ['SPY'] + all_stocks:  # SPY + 25개 종목 순회
    try:  # 예외 처리
        t = yf.Ticker(ticker)  # 야후 파이낸스 티커 객체 생성
        hist = t.history(start=start_date, auto_adjust=True)  # 과거 데이터 수집
        if len(hist) > 0:  # 데이터가 있으면
            stock_frames[ticker] = hist['Close']  # 종가 저장
            print(f'  ✓ {ticker}: {len(hist)}일')  # 수집 성공 메시지
        else:  # 데이터 없으면
            print(f'  ✗ {ticker}: 데이터 없음')  # 실패 메시지
    except Exception as e:  # 예외 발생 시
        print(f'  ✗ {ticker}: {e}')  # 에러 메시지

stock_prices = pd.DataFrame(stock_frames)  # 종목별 종가 데이터프레임 생성
stock_prices.index = pd.to_datetime(stock_prices.index)  # 인덱스를 날짜형으로 변환
if stock_prices.index.tz is not None:  # 타임존이 있으면
    stock_prices.index = stock_prices.index.tz_localize(None)  # 타임존 제거

# 일별 수익률 계산
stock_returns = stock_prices.___().___()  # 일별 퍼센트 변화율 + 결측 제거 (Q5-1, Q5-2)
spy_ret = stock_returns['SPY']  # SPY 수익률 분리

print(f'\n수익률 데이터: {stock_returns.shape}')  # 결과 출력

In [ ]:
# ── Cell 5-B: 베타 제거 잔차 수익률 ──────────────────────────────────────
print('시장 베타 제거 중...')  # 베타 제거 시작 메시지
residuals = pd.DataFrame(index=stock_returns.index)  # 잔차 저장용 데이터프레임

for ticker in all_stocks:  # 25개 종목에 대해
    if ticker not in stock_returns.columns:  # 데이터가 없으면 건너뜀
        continue  # 다음 종목으로
    ret = stock_returns[ticker]  # 해당 종목 수익률
    # 60일 롤링 베타 = Cov(종목, SPY) / Var(SPY)
    cov_with_spy = ret.___(60, min_periods=30).___(spy_ret)  # 60일 롤링 공분산 (Q5-3, Q5-4)
    spy_var = spy_ret.___(60, min_periods=30).___()  # 60일 롤링 분산 (Q5-5, Q5-6)
    beta = cov_with_spy / spy_var  # 베타 계산 (공분산/분산)
    residuals[ticker] = ret - beta * spy_ret  # 잔차 = 수익률 - 베타 × 시장수익률

residuals = residuals.dropna(how='all')  # 전부 NaN인 행 제거
print(f'잔차 데이터: {residuals.shape}')  # 결과 출력

In [ ]:
# ── Cell 5-C: 잔차 페어와이즈 상관 + 디스퍼전 ────────────────────────────
def sector_residual_corr(resid_df, sector_dict, window=20):  # 섹터별 잔차 상관 함수
    """섹터별 잔차 20일 롤링 페어와이즈 상관 → 전체 평균."""  # 독스트링
    sector_corrs = []  # 섹터별 상관 저장 리스트
    for sector, stocks in sector_dict.items():  # 각 섹터에 대해
        available = [s for s in stocks if s in resid_df.columns]  # 사용 가능한 종목 필터
        if len(available) < 2:  # 2개 미만이면 건너뜀
            continue  # 다음 섹터로
        pairs = list(combinations(available, 2))  # 모든 종목 쌍 생성
        pair_corrs = []  # 쌍별 상관 저장 리스트
        for s1, s2 in pairs:  # 각 쌍에 대해
            rc = resid_df[s1].___(window).___(resid_df[s2])  # 20일 롤링 상관 계산 (Q5-7, Q5-8)
            pair_corrs.append(rc)  # 결과 추가
        sector_avg = pd.concat(pair_corrs, axis=1).___(___)  # 쌍별 평균 계산 (Q5-9)
        sector_corrs.append(sector_avg)  # 섹터 평균 추가
    return pd.concat(sector_corrs, axis=1).___(___)  # 전체 섹터 평균 반환 (Q5-10)

daily_resid_corr = sector_residual_corr(residuals, SECTOR_STOCKS).___()  # 일별 잔차 상관 + 결측 제거 (Q5-11)
residual_corr_monthly = daily_resid_corr.resample('MS').___().___()  # 월별 리샘플링 (Q5-12, Q5-13)
residual_corr_monthly.name = 'residual_corr'  # 시리즈 이름 설정

# ── 크로스섹션 디스퍼전: 전 종목 일별 수익률의 횡단면 표준편차 ──
available_stocks = [s for s in all_stocks if s in stock_returns.columns]  # 사용 가능 종목 필터
cross_section_std = stock_returns[available_stocks].___(axis=1)  # 매일 횡단면 표준편차 (Q5-14)
dispersion_daily = cross_section_std.___(20).___().___()  # 20일 롤링 평균 + 결측 제거 (Q5-15, Q5-16, Q5-17)
dispersion_monthly = dispersion_daily.resample('MS').___().___()  # 월별 리샘플링 (Q5-18, Q5-19)
dispersion_monthly.name = 'dispersion'  # 시리즈 이름 설정

print(f'변수 ③-A 잔차 상관: {len(residual_corr_monthly)}개월')  # 결과 출력
print(f'변수 ③-B 디스퍼전: {len(dispersion_monthly)}개월')  # 결과 출력

In [ ]:
# ── Cell 5-D: 섹터 동조화 시각화 ─────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)  # 2행 서브플롯

axes[0].plot(residual_corr_monthly.index, residual_corr_monthly, color='#9C27B0', alpha=0.8)  # 잔차 상관 플롯
axes[0].fill_between(residual_corr_monthly.index, residual_corr_monthly, residual_corr_monthly.mean(),
                alpha=0.2, color='#9C27B0')  # 평균 대비 영역 채움
axes[0].axhline(residual_corr_monthly.mean(), color='gray', linestyle='--',
               label=f'평균 ({residual_corr_monthly.mean():.3f})')  # 평균선
axes[0].set_title('③-A 잔차 상관 (베타 제거 후, 높을수록 감정적)')  # 제목
axes[0].set_ylabel('상관계수')  # Y축 라벨
axes[0].legend()  # 범례
axes[0].grid(alpha=0.3)  # 격자

axes[1].plot(dispersion_monthly.index, dispersion_monthly, color='#00BCD4', alpha=0.8)  # 디스퍼전 플롯
axes[1].fill_between(dispersion_monthly.index, dispersion_monthly, dispersion_monthly.mean(),
                alpha=0.2, color='#00BCD4')  # 평균 대비 영역 채움
axes[1].axhline(dispersion_monthly.mean(), color='gray', linestyle='--',
               label=f'평균 ({dispersion_monthly.mean():.4f})')  # 평균선
axes[1].set_title('③-B 크로스섹션 디스퍼전 (낮을수록 감정적/묻지마 매매)')  # 제목
axes[1].set_ylabel('디스퍼전')  # Y축 라벨
axes[1].legend()  # 범례
axes[1].grid(alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

## Q6. 변수 ④ Amihud 비유동성 (로그 + 윈저라이징)
- `np.log()`, `.abs()`, `.clip()`, `.rolling()`, `.mean()`, `.quantile()` 등을 채우세요

In [ ]:
# ── Cell 6-A: Amihud 비유동성 데이터 수집 + 계산 ─────────────────────────
AMIHUD_STOCKS = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META']  # 대형주 5종목

print(f'Amihud 비유동성: {len(AMIHUD_STOCKS)}개 대형주 OHLCV 수집 중...')  # 수집 시작
amihud_frames = {}  # OHLCV 데이터 저장 딕셔너리
for ticker in AMIHUD_STOCKS:  # 각 종목에 대해
    try:  # 예외 처리
        t = yf.Ticker(ticker)  # 티커 객체 생성
        hist = t.history(start=start_date, auto_adjust=True)  # 과거 데이터 수집
        if len(hist) > 0:  # 데이터가 있으면
            amihud_frames[ticker] = hist[['Open', 'Close', 'Volume']]  # OHLCV 저장
            print(f'  ✓ {ticker}: {len(hist)}일')  # 수집 성공
        else:  # 데이터 없으면
            print(f'  ✗ {ticker}: 데이터 없음')  # 실패
    except Exception as e:  # 예외 발생 시
        print(f'  ✗ {ticker}: {e}')  # 에러 메시지

# 종목별 Amihud = |log(close/open)| / (close × volume)
amihud_per_stock = []  # 종목별 Amihud 저장 리스트
for ticker, df_t in amihud_frames.items():  # 각 종목에 대해
    oc_ret = ___(df_t['Close'] / df_t['Open']).___()  # open-to-close 절대 로그수익률 (Q6-1, Q6-2)
    dollar_vol = df_t['Close'] * df_t['Volume']  # 달러 거래량 계산
    amihud_t = oc_ret / dollar_vol.replace(0, np.nan)  # Amihud 비율 (0 거래량 → NaN)
    amihud_per_stock.append(amihud_t)  # 결과 추가

# 5종목 평균 → 20일 롤링 평균
amihud_avg = pd.concat(amihud_per_stock, axis=1).___(___).___(___) # 종목 평균 + 결측 제거 (Q6-3, Q6-4)
amihud_rolling = amihud_avg.___(20).___().___()  # 20일 롤링 평균 + 결측 제거 (Q6-5, Q6-6, Q6-7)

# 로그 변환
log_amihud = ___(amihud_rolling + 1e-15)  # 로그 변환 (epsilon 추가) (Q6-8)

# 1/99 퍼센타일 윈저라이징
q01 = log_amihud.quantile(___)  # 1% 퍼센타일 (Q6-9)
q99 = log_amihud.quantile(___)  # 99% 퍼센타일 (Q6-10)
log_amihud_w = log_amihud.___(q01, q99)  # 범위 밖 값 클리핑 (Q6-11)

# 타임존 제거
if hasattr(log_amihud_w.index, 'tz') and log_amihud_w.index.tz is not None:  # 타임존 존재 확인
    log_amihud_w.index = log_amihud_w.index.tz_localize(None)  # 타임존 제거

# 월별 리샘플링
amihud_monthly = log_amihud_w.resample('MS').___().___()  # 월별 평균 + 결측 제거 (Q6-12, Q6-13)
amihud_monthly.name = 'amihud'  # 시리즈 이름 설정

print(f'\n변수 ④ Amihud 비유동성: {len(amihud_monthly)}개월')  # 관측 수 출력

In [ ]:
# ── Cell 6-B: Amihud 시각화 ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))  # 단일 플롯 생성
ax.plot(amihud_monthly.index, amihud_monthly, color='#E91E63', alpha=0.8)  # Amihud 플롯
ax.fill_between(amihud_monthly.index, amihud_monthly, amihud_monthly.mean(), alpha=0.2, color='#E91E63')  # 영역 채움
ax.axhline(amihud_monthly.mean(), color='gray', linestyle='--', label=f'평균 ({amihud_monthly.mean():.2f})')  # 평균선
ax.set_title('④ Amihud 비유동성 (로그 + 윈저라이징, 높을수록 스트레스)')  # 제목
ax.set_ylabel('log(Amihud)')  # Y축 라벨
ax.legend()  # 범례
ax.grid(alpha=0.3)  # 격자
plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

## Q7. 변수 ⑤ VIX 기간구조 + 변수 ⑥ 하이일드 스프레드
- VIX / VIX3M 비율 계산, HY OAS 복사

In [ ]:
# ── Cell 7: 변수 ⑤ VIX 기간구조 ────────────────────────────────────────────
# VIX / VIX3M 비율: > 1.0 = 백워데이션 (단기 공포 > 장기) = 패닉 신호
vix_term_df = pd.DataFrame({  # VIX와 VIX3M을 하나의 데이터프레임으로 결합
    'vix': vix_monthly,  # VIX 월별 데이터
    'vix3m': vix3m_monthly,  # VIX3M 월별 데이터
}).___()  # 결측 행 제거 (Q7-1)

vix_term = vix_term_df['vix'] / vix_term_df['vix3m']  # VIX / VIX3M 비율 계산
vix_term = vix_term.___()  # 결측 제거 (Q7-2)
vix_term.name = 'vix_term'  # 시리즈 이름 설정

print(f'변수 ⑤ VIX 기간구조 (VIX/VIX3M): {len(vix_term)}개월')  # 관측 수 출력
print(f'  평균: {vix_term.___():.4f}  |  표준편차: {vix_term.___():.4f}')  # 기술통계 (Q7-3, Q7-4)
print(f'  백워데이션(>1.0) 비율: {(vix_term > 1.0).___() *100:.1f}%')  # 백워데이션 비율 (Q7-5)

# 시각화
fig, ax = plt.subplots(figsize=(14, 4))  # 단일 플롯
ax.plot(vix_term.index, vix_term, color='#FF5722', alpha=0.8)  # VIX 기간구조 플롯
ax.fill_between(vix_term.index, vix_term, 1.0,
                where=vix_term > 1.0, color='#F44336', alpha=0.3, label='백워데이션 (패닉)')  # 백워데이션 영역
ax.fill_between(vix_term.index, vix_term, 1.0,
                where=vix_term <= 1.0, color='#4CAF50', alpha=0.2, label='콘탱고 (정상)')  # 콘탱고 영역
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)  # 기준선
ax.set_title('⑤ VIX 기간구조 (VIX/VIX3M, >1.0 = 패닉)')  # 제목
ax.set_ylabel('VIX / VIX3M')  # Y축 라벨
ax.legend()  # 범례
ax.grid(alpha=0.3)  # 격자
plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

In [ ]:
# ── Cell 8: 변수 ⑥ 하이일드 스프레드 ──────────────────────────────────────
hy_spread = hy_monthly.copy()  # 하이일드 OAS 복사
hy_spread.name = 'hy_spread'  # 시리즈 이름 설정

print(f'변수 ⑥ 하이일드 스프레드: {len(hy_spread)}개월')  # 관측 수 출력
print(f'  평균: {hy_spread.___():.4f}  |  표준편차: {hy_spread.___():.4f}')  # 기술통계 (Q7-6, Q7-7)

# 시각화
fig, ax = plt.subplots(figsize=(14, 4))  # 단일 플롯
ax.plot(hy_spread.index, hy_spread, color='#795548', alpha=0.8)  # HY OAS 플롯
ax.fill_between(hy_spread.index, hy_spread, hy_spread.mean(), alpha=0.2, color='#795548')  # 영역 채움
ax.axhline(hy_spread.mean(), color='gray', linestyle='--',
           label=f'평균 ({hy_spread.mean():.2f})')  # 평균선
ax.axhline(5.0, color='red', linestyle=':', alpha=0.5, label='위기 임계 (5%)')  # 위기 기준선
ax.set_title('⑥ ICE BofA US High Yield OAS (높을수록 시스템 리스크)')  # 제목
ax.set_ylabel('OAS (%)')  # Y축 라벨
ax.legend()  # 범례
ax.grid(alpha=0.3)  # 격자
plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

## Q8. 피처 병합 + 윈저라이징 + 상관분석
- 변수 ⑦ 실현 변동성 계산, 8개 피처를 하나의 DataFrame으로 병합
- `np.sqrt()`, `.rolling()`, `.std()`, `.clip()`, `.quantile()`, `.corr()` 등을 채우세요

In [ ]:
# ── Cell 9-A: 변수 ⑦ 실현 변동성 + 피처 병합 ────────────────────────────
def strip_tz(s):  # 타임존 제거 함수
    s = s.copy()  # 복사본 생성
    if hasattr(s.index, 'tz') and s.index.tz is not None:  # 타임존 존재 확인
        s.index = s.index.tz_localize(None)  # 타임존 제거
    return s  # 반환

# ── 변수 ⑦ S&P 500 실현 변동성 (20일 롤링 연율화) ──
spy_ret_clean = strip_tz(spy_ret)  # SPY 수익률 타임존 제거
realized_vol_daily = spy_ret_clean.___(20).___() * ___(___) # 20일 롤링 표준편차 × √252 연율화 (Q8-1, Q8-2, Q8-3, Q8-4)
realized_vol_monthly = realized_vol_daily.resample('MS').___().___()  # 월별 평균 + 결측 제거 (Q8-5, Q8-6)
realized_vol_monthly.name = 'realized_vol'  # 시리즈 이름 설정
print(f'변수 ⑦ 실현 변동성: {len(realized_vol_monthly)}개월')  # 관측 수 출력

# 8개 시리즈를 딕셔너리로 모음
all_series = {
    'fundamental_gap': fundamental_gap,  # ① 펀더멘털 괴리
    'erp_zscore':      erp_zscore,  # ② ERP Z-Score
    'residual_corr':   residual_corr_monthly,  # ③-A 잔차 상관
    'dispersion':      dispersion_monthly,  # ③-B 디스퍼전
    'amihud':          amihud_monthly,  # ④ Amihud
    'vix_term':        vix_term,  # ⑤ VIX 기간구조
    'hy_spread':       hy_spread,  # ⑥ HY 스프레드
    'realized_vol':    realized_vol_monthly,  # ⑦ 실현 변동성
}

print('\n각 시리즈 마지막 날짜:')  # 시리즈 정보 출력
for name, s in all_series.items():  # 각 시리즈에 대해
    last = s.___().index[-1]  # 마지막 유효 날짜 (Q8-7)
    print(f'  {name:20s}: {last}  ({len(s.dropna())}개월)')  # 결과 출력

# 데이터프레임으로 병합
features = pd.DataFrame({k: strip_tz(v) for k, v in all_series.items()})  # 타임존 제거 후 병합
features = features.___()  # 결측 행 제거 (Q8-8)
print(f'\n피처 DataFrame: {features.shape}')  # 결과 출력

In [ ]:
# ── Cell 9-B: 윈저라이징 + 상관 히트맵 ───────────────────────────────────
# ── 윈저라이징 (1/99 퍼센타일) — 극단값 영향 제한 ──
print(f'윈저라이징 전: {features.shape}')  # 윈저라이징 전 shape 출력
for col in FEATURE_NAMES:  # 8개 피처에 대해
    q01 = features[col].quantile(___)  # 1% 퍼센타일 (Q8-9)
    q99 = features[col].quantile(___)  # 99% 퍼센타일 (Q8-10)
    n_clipped = ((features[col] < q01) | (features[col] > q99)).sum()  # 클리핑 대상 수
    features[col] = features[col].___(q01, q99)  # 범위 밖 값 클리핑 (Q8-11)
    if n_clipped > 0:  # 클리핑된 값이 있으면
        print(f'  {col}: {n_clipped}개 값 클리핑 [{q01:.4f}, {q99:.4f}]')  # 결과 출력
print(f'윈저라이징 후: {features.shape}')  # 윈저라이징 후 shape 출력

print(f'\n기술통계:')  # 기술통계 출력
print(features.describe().round(4))  # describe 결과 출력

# 상관 히트맵
fig, ax = plt.subplots(figsize=(10, 8))  # 히트맵 플롯 생성
corr = features.___()  # 상관행렬 계산 (Q8-12)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # 상삼각 마스크
feature_display = ['① 펀더멘털괴리', '② ERP Z-Score', '③-A 잔차상관',
                   '③-B 디스퍼전', '④ Amihud', '⑤ VIX기간구조', '⑥ HY스프레드',
                   '⑦ 실현변동성']  # 표시용 피처 이름
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    mask=mask, ax=ax, vmin=-1, vmax=1,
    xticklabels=feature_display,
    yticklabels=feature_display,
)  # 히트맵 그리기
ax.set_title('8개 변수 상관행렬')  # 제목
plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

## Q9. HMM 학습 (GaussianHMM, 4상태)
- `RobustScaler`, `GaussianHMM` 클래스와 파라미터(`n_components`, `covariance_type`, `n_iter`)를 채우세요
- `.fit_transform()`, `.fit()`, `.predict()`, `.score()` 메서드를 채우세요

In [ ]:
# ── Cell 10: HMM 학습 (N_STATES=4) ───────────────────────────────────────
X = features[FEATURE_NAMES].values  # 피처 값을 numpy 배열로 추출
scaler = ___()  # 로버스트 스케일러 인스턴스 생성 (Q9-1)
X_scaled = scaler.___(X)  # 스케일링 학습 + 변환 동시 수행 (Q9-2)

# GaussianHMM: full 공분산 시도 → diag 폴백
for cov_type in ('___', '___'):  # 공분산 타입 순서: full → diag (Q9-3, Q9-4)
    try:  # 예외 처리
        model = ___(  # HMM 가우시안 모델 생성 (Q9-5)
            n_components=___,  # 상태 수 (Q9-6)
            covariance_type=cov_type,  # 공분산 타입
            n_iter=___,  # 반복 횟수: 200 (Q9-7)
            random_state=42,  # 재현성을 위한 시드
        )
        model.___(X_scaled)  # 모델 학습 (Q9-8)
        print(f'HMM 학습 완료 (covariance: {cov_type}, scaler: RobustScaler)')  # 완료 메시지
        break  # 성공하면 루프 탈출
    except (ValueError, np.linalg.LinAlgError):  # 행렬 에러 발생 시
        if cov_type == 'diag':  # diag에서도 실패하면
            raise  # 예외 발생

# Viterbi 상태 시퀀스
states = model.___(X_scaled)  # 최적 상태 시퀀스 예측 (Q9-9)
features['state'] = states  # 상태 컬럼 추가

ll = model.___(X_scaled)  # 로그 우도 점수 계산 (Q9-10)
print(f'Log-likelihood: {ll:.2f}')  # 로그 우도 출력
print(f'Per-sample LL: {ll / len(X_scaled):.4f}')  # 샘플당 로그 우도 출력
print(f'학습 데이터: {len(X_scaled)}개월, 피처: {X_scaled.shape[1]}개')  # 데이터 크기 출력
print(f'\n상태별 관측 수:')  # 상태별 관측 수 출력
print(pd.Series(states).value_counts().sort_index())  # 상태별 카운트

## Q10. noise_score 기반 상태 매핑 + 시각화
- noise_score 가중치(0.5, 0.3, 1.0, 0.0, 0.5, 2.0, 1.5, 2.0)를 채우세요
- `np.abs()`, `np.argsort()` 등을 채우세요

In [ ]:
# ── Cell 11-A: noise_score 함수 정의 + 상태 매핑 ─────────────────────────
def compute_noise_score(means):  # noise_score v2 계산 함수
    """피처 순서: fundamental_gap, erp_zscore, residual_corr,
              dispersion, amihud, vix_term, hy_spread, realized_vol"""  # 독스트링
    return (
        ___ * np.___( means[:, 0])   # fundamental_gap 가중치 × 절대값 (Q10-1, Q10-2)
      + ___ * np.___(means[:, 1])   # erp_zscore 가중치 × 절대값 (Q10-3, Q10-4)
      + ___ * means[:, 2]           # residual_corr 가중치 (Q10-5)
      # dispersion 제거 (가중치 0)
      + ___ * means[:, 4]           # amihud 가중치 (Q10-6)
      + ___ * means[:, 5]           # vix_term 가중치 (가장 유의미) (Q10-7)
      + ___ * means[:, 6]           # hy_spread 가중치 (Q10-8)
      + ___ * means[:, 7]           # realized_vol 가중치 (핵심) (Q10-9)
    )

noise_scores = compute_noise_score(model.means_)  # 각 상태의 noise_score 계산
sorted_states = np.___(noise_scores)  # noise_score 오름차순 정렬 인덱스 (Q10-10)
state_to_phase = {int(sid): RANK_TO_PHASE[rank] for rank, sid in enumerate(sorted_states)}  # 상태→국면 매핑
features['phase'] = [state_to_phase[s] for s in features['state']]  # 국면 컬럼 추가

print('noise_score v2 가중치:')  # 가중치 출력
weights = [0.5, 0.3, 1.0, 0.0, 0.5, 2.0, 1.5, 2.0]  # 가중치 리스트
feature_labels = ['펀더멘털괴리', 'ERP Z-Score', '잔차상관', '디스퍼전',
                  'Amihud', 'VIX기간구조', 'HY스프레드', '실현변동성']  # 피처 표시명
for fl, w in zip(feature_labels, weights):  # 피처와 가중치를 함께 순회
    if w > 0:  # 가중치가 있으면
        print(f'  {fl:12s}: {w:.1f}')  # 가중치 출력
    else:  # 가중치가 0이면
        print(f'  {fl:12s}: 제거')  # 제거 표시

print('\nHMM 상태 → 국면 매핑 (noise_score 오름차순):')  # 매핑 결과 출력
for rank, sid in enumerate(sorted_states):  # 정렬된 상태를 순회
    ph = RANK_TO_PHASE[rank]  # 국면 번호
    ns = noise_scores[sid]  # noise_score
    means_orig = scaler.inverse_transform(model.means_[[sid]])[0]  # 원본 스케일로 역변환
    print(f'  state {sid} → {PHASE_NAMES[ph]}  (noise_score: {ns:.3f})')  # 매핑 출력
    for i, fl in enumerate(feature_labels):  # 각 피처에 대해
        print(f'    {fl}: {means_orig[i]:.4f} (scaled: {model.means_[sid, i]:.4f})')  # 값 출력

In [ ]:
# ── Cell 11-B: 국면 분포 시각화 ──────────────────────────────────────────
from matplotlib.patches import Patch  # 범례 패치 임포트

fig, axes = plt.subplots(1, 2, figsize=(14, 4))  # 1행 2열 서브플롯

counts = features['phase'].value_counts().sort_index()  # 국면별 관측 수
colors = [PHASE_COLORS[i] for i in counts.index]  # 국면별 색상
axes[0].bar([PHASE_NAMES[i] for i in counts.index], counts.values, color=colors)  # 바 차트
axes[0].set_title('국면별 관측 횟수')  # 제목
axes[0].set_ylabel('개월 수')  # Y축 라벨
for i, v in enumerate(counts.values):  # 각 바에
    axes[0].text(i, v + 0.5, str(v), ha='center', fontsize=11)  # 값 표시

sp_sorted = shiller['P'].sort_index()  # S&P 500 가격 정렬
sp_aligned = sp_sorted.reindex(features.index, method='ffill')  # 피처 인덱스에 맞춤
axes[1].scatter(features.index, sp_aligned,
               c=[PHASE_COLORS[p] for p in features['phase']], s=20, alpha=0.8)  # 산점도
axes[1].set_title('S&P 500 (색상 = 국면)')  # 제목
axes[1].set_ylabel('S&P 500')  # Y축 라벨

legend_elements = [Patch(facecolor=PHASE_COLORS[k], label=PHASE_NAMES[k]) for k in range(4)]  # 범례 요소
fig.legend(handles=legend_elements, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))  # 범례 추가

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

print('\n국면 분포:')  # 국면 분포 출력
for ph, cnt in counts.items():  # 각 국면에 대해
    print(f'  {PHASE_NAMES[ph]}: {cnt}개월 ({cnt/len(features)*100:.1f}%)')  # 비율 출력

## Q11. 전이확률 행렬 + 지속기간
- `model.transmat_`에서 전이확률을 추출하고 평균 지속기간을 계산하세요

In [ ]:
# ── Cell 12: 전이확률 행렬 + 지속기간 ──────────────────────────────────────
trans_mat = model.transmat_  # 전이확률 행렬 추출

# 국면 순서로 재배열
phase_order = []  # 국면 순서 리스트
for ph in range(4):  # 각 국면에 대해
    for s in range(4):  # 각 상태에 대해
        if state_to_phase[s] == ph:  # 매핑 일치 확인
            phase_order.append(s)  # 순서 추가
            break  # 다음 국면으로

reordered = trans_mat[np.ix_(phase_order, phase_order)]  # 국면 순서로 행렬 재배열
reordered_labels = [PHASE_NAMES[state_to_phase[s]] for s in phase_order]  # 재배열된 라벨

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1행 2열 서브플롯

# [좌] 전이 확률 히트맵
sns.heatmap(
    reordered, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=reordered_labels, yticklabels=reordered_labels,
    ax=axes[0], vmin=0, vmax=1, cbar_kws={'label': '전이 확률'}
)  # 히트맵 시각화
axes[0].set_title('국면 전이 확률 행렬')  # 제목
axes[0].set_xlabel('다음 국면')  # X축 라벨
axes[0].set_ylabel('현재 국면')  # Y축 라벨

# [우] 국면별 관측 수 + 평균 지속기간
counts_ordered = [int((features['phase'] == ph).sum()) for ph in range(4)]  # 국면별 관측 수
durations = [1 / (1 - reordered[i, i]) if reordered[i, i] < 1 else float('inf') for i in range(4)]  # 평균 지속기간 = 1/(1-자기전이확률)

bars = axes[1].bar(
    [PHASE_NAMES[ph] for ph in range(4)], counts_ordered,
    color=[PHASE_COLORS[ph] for ph in range(4)], alpha=0.8
)  # 바 차트
axes[1].set_title('국면별 관측 횟수 및 평균 지속기간')  # 제목
axes[1].set_ylabel('개월 수')  # Y축 라벨
for i, (v, d) in enumerate(zip(counts_ordered, durations)):  # 각 바에
    axes[1].text(i, v + 0.5, f'{v}개월\n(평균 {d:.0f}개월)', ha='center', fontsize=9)  # 값 표시

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

# 전이 확률 요약
print('전이 확률 요약:')  # 전이 확률 요약 출력
for i, ph in enumerate(range(4)):  # 각 국면에 대해
    s = phase_order[i]  # 해당 상태 번호
    self_prob = trans_mat[s, s]  # 자기 전이 확률
    print(f'  {PHASE_NAMES[ph]}: 자기전이 {self_prob:.1%} → 평균 지속 {1/(1-self_prob):.1f}개월')  # 결과 출력

## Q12. 발산 분포 분석 (국면별 변수 특성)
- `scaler.inverse_transform()`을 사용하여 원본 스케일로 복원합니다

In [ ]:
# ── Cell 13: 발산 분포 분석 (국면별 변수 특성) ──────────────────────────────
means_reordered = scaler.___(model.means_[phase_order])  # 원본 스케일로 역변환 (Q12-1)
feature_display = ['① 펀더멘털괴리', '② ERP Z', '③-A 잔차상관', '③-B 디스퍼전',
                   '④ Amihud', '⑤ VIX기간구조', '⑥ HY스프레드', '⑦ 실현변동성']  # 표시명

fig, axes = plt.subplots(2, 1, figsize=(15, 9))  # 2행 서브플롯

x = np.arange(len(FEATURE_NAMES))  # x축 위치
width = 0.18  # 바 너비
for i, ph in enumerate(range(4)):  # 각 국면에 대해
    axes[0].bar(x + i * width - 1.5 * width, means_reordered[i], width,
                label=PHASE_NAMES[ph], color=PHASE_COLORS[ph], alpha=0.85)  # 바 차트
axes[0].set_xticks(x)  # x축 눈금
axes[0].set_xticklabels(feature_display, fontsize=9, rotation=15)  # x축 라벨
axes[0].set_title('국면별 변수 평균값 (HMM Emission Means, 원본 스케일)')  # 제목
axes[0].legend()  # 범례
axes[0].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[0].grid(axis='y', alpha=0.3)  # 격자

key_features = ['fundamental_gap', 'residual_corr', 'hy_spread', 'realized_vol']  # 주요 피처
key_labels = ['① 펀더멘털괴리', '③-A 잔차상관', '⑥ HY스프레드', '⑦ 실현변동성']  # 주요 피처 라벨

for idx, (feat, label) in enumerate(zip(key_features, key_labels)):  # 주요 피처에 대해
    for ph in range(4):  # 각 국면에 대해
        mask = features['phase'] == ph  # 국면 마스크
        vals = features.loc[mask, feat].values  # 해당 값 추출
        pos = idx * 5 + ph  # 박스플롯 위치
        axes[1].boxplot(vals, positions=[pos], widths=0.6, patch_artist=True,
                        boxprops=dict(facecolor=PHASE_COLORS[ph], alpha=0.7),
                        medianprops=dict(color='black'), showfliers=False)  # 박스플롯
    if idx < len(key_features) - 1:  # 마지막이 아니면
        axes[1].axvline(idx * 5 + 3.5, color='gray', linewidth=0.5, linestyle='--')  # 구분선

axes[1].set_xticks([1.5, 6.5, 11.5, 16.5])  # x축 눈금
axes[1].set_xticklabels(key_labels)  # x축 라벨
axes[1].set_title('주요 변수의 국면별 분포')  # 제목
axes[1].set_ylabel('값')  # Y축 라벨

legend_elements = [Patch(facecolor=PHASE_COLORS[k], label=PHASE_NAMES[k]) for k in range(4)]  # 범례 요소
axes[1].legend(handles=legend_elements, loc='upper right')  # 범례

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

print('\n국면별 주요 변수 평균:')  # 결과 테이블 출력
header = f'{"":20s} {"펀더멘털":>8s} {"ERP Z":>8s} {"잔차상관":>8s} {"디스퍼전":>8s} {"Amihud":>8s} {"VIX기간":>8s} {"HY OAS":>8s} {"실현Vol":>8s}'
print(header)  # 헤더 출력
print('-' * len(header))  # 구분선
for i, ph in enumerate(range(4)):  # 각 국면에 대해
    m = means_reordered[i]  # 평균값 추출
    print(f'{PHASE_NAMES[ph]:20s} {m[0]:8.3f} {m[1]:8.3f} {m[2]:8.4f} {m[3]:8.4f} {m[4]:8.2f} {m[5]:8.3f} {m[6]:8.2f} {m[7]:8.3f}')  # 값 출력

## Q13. 일별 국면 예측
- 최신 피처 값을 구하고 `scaler.transform()`, `model.predict_proba()`로 국면 확률을 예측하세요
- `np.sqrt()`, `.corr()`, `.std()`, `.mean()` 등을 채우세요

In [ ]:
# ── Cell 14-A: 일별 피처 값 계산 ──────────────────────────────────────────
import datetime as _dt  # datetime 별칭 임포트
from itertools import combinations as _combinations  # combinations 별칭 임포트

today = pd.Timestamp(_dt.date.today())  # 오늘 날짜 타임스탬프
print(f'일별 국면 예측 — 기준일: {today.strftime("%Y-%m-%d")}')  # 기준일 출력
print('=' * 60)  # 구분선

# ── ① fundamental_gap: 월별 → forward-fill ──
fg_val = features['fundamental_gap'].iloc[-1]  # 마지막 월 값

# ── ② erp_zscore: 월별 → forward-fill ──
ez_val = features['erp_zscore'].iloc[-1]  # 마지막 월 값

# ── ③-A residual_corr: 최근 20일 잔차 상관 평균 ──
recent_resid = residuals.iloc[-20:]  # 최근 20영업일 잔차
pair_corrs = []  # 쌍별 상관 리스트
for sector, stocks in SECTOR_STOCKS.items():  # 각 섹터에 대해
    avail = [s for s in stocks if s in recent_resid.columns]  # 사용 가능 종목
    if len(avail) < 2:  # 2개 미만이면 건너뜀
        continue  # 다음 섹터
    for s1, s2 in _combinations(avail, 2):  # 각 쌍에 대해
        c = recent_resid[s1].___(recent_resid[s2])  # 쌍별 상관 계산 (Q13-1)
        pair_corrs.append(c)  # 결과 추가
rc_val = float(np.___(pair_corrs)) if pair_corrs else residual_corr_monthly.iloc[-1]  # 평균 (Q13-2)

# ── ③-B dispersion: 최근 20일 횡단면 std 평균 ──
recent_ret = stock_returns[available_stocks].iloc[-20:]  # 최근 20일 수익률
disp_val = float(recent_ret.___(axis=1).___())  # 횡단면 표준편차의 평균 (Q13-3, Q13-4)

# ── ④ amihud: 최근 20일 → log → winsorize ──
amihud_daily_vals = []  # 일별 Amihud 리스트
for ticker, df_t in amihud_frames.items():  # 각 종목에 대해
    recent_t = df_t.iloc[-20:]  # 최근 20일 데이터
    oc_ret = ___(recent_t['Close'] / recent_t['Open']).___()  # 절대 로그수익률 (Q13-5, Q13-6)
    dollar_vol = recent_t['Close'] * recent_t['Volume']  # 달러 거래량
    ami_t = oc_ret / dollar_vol.replace(0, np.nan)  # Amihud 비율
    amihud_daily_vals.append(ami_t)  # 결과 추가
amihud_recent = pd.concat(amihud_daily_vals, axis=1).___(axis=1).___()  # 종목 평균 + 결측 제거 (Q13-7, Q13-8)
log_ami = ___(amihud_recent.___() + 1e-15)  # 로그 변환 (Q13-9, Q13-10)
ami_val = float(np.___(log_ami, q01, q99))  # 윈저라이징 (Q13-11)

In [ ]:
# ── Cell 14-B: 나머지 피처 + 국면 예측 ────────────────────────────────────
# ── ⑤ vix_term: 최신 일별 VIX / VIX3M ──
latest_vix = df_vix['vix'].___().iloc[-1]  # 최신 VIX 값 (Q13-12)
latest_vix3m = df_vix3m['vix3m'].___().iloc[-1]  # 최신 VIX3M 값 (Q13-13)
vt_val = float(latest_vix / latest_vix3m)  # VIX/VIX3M 비율

# ── ⑥ hy_spread: 최신 일별 값 ──
hy_val = float(df_hy['hy_spread'].___().iloc[-1])  # 최신 HY OAS 값 (Q13-14)

# ── ⑦ realized_vol: 최근 20일 SPY 실현 변동성 (연율화) ──
spy_ret_recent = spy_ret.iloc[-20:]  # 최근 20일 SPY 수익률
rv_val = float(spy_ret_recent.___() * ___(___))  # 표준편차 × √252 연율화 (Q13-15, Q13-16, Q13-17)

# ── 8개 피처 → 예측 ──
daily_features = np.array([[fg_val, ez_val, rc_val, disp_val, ami_val, vt_val, hy_val, rv_val]])  # 8개 피처 배열
daily_scaled = scaler.___(daily_features)  # 스케일링 변환 (Q13-18)
daily_proba_raw = model.___(daily_scaled)[0]  # 국면 확률 예측 (Q13-19)

proba_by_phase = {state_to_phase[i]: float(p) for i, p in enumerate(daily_proba_raw)}  # 국면별 확률 매핑
pred_phase = max(proba_by_phase, key=proba_by_phase.get)  # 최고 확률 국면

# ── 결과 출력 ──
print(f'\n  현재 국면: {PHASE_NAMES[pred_phase]}  (확률 {proba_by_phase[pred_phase]*100:.0f}%)')  # 현재 국면 출력
print()  # 빈 줄
print('  국면별 확률:')  # 국면별 확률 출력 시작
for ph in range(4):  # 각 국면에 대해
    prob = proba_by_phase.get(ph, 0.0)  # 확률 추출
    bar = '█' * int(prob * 30)  # 막대 그래프
    print(f'    {PHASE_NAMES[ph]:20s} {prob*100:5.1f}% {bar}')  # 결과 출력

print()  # 빈 줄
print('  오늘 변수 값:')  # 변수 값 출력
print(f'    ① 펀더멘털 괴리:       {fg_val:+.4f}  (월별 ffill)')  # ① 값
print(f'    ② ERP Z-Score:         {ez_val:.4f}  (월별 ffill)')  # ② 값
print(f'    ③-A 잔차 상관:         {rc_val:.4f}  (최근 20일)')  # ③-A 값
print(f'    ③-B 디스퍼전:          {disp_val:.4f}  (최근 20일)')  # ③-B 값
print(f'    ④ Amihud 비유동성:     {ami_val:.4f}  (최근 20일)')  # ④ 값
print(f'    ⑤ VIX 기간구조:       {vt_val:.4f}  (일별)')  # ⑤ 값
print(f'    ⑥ HY 스프레드:        {hy_val:.4f}  (일별)')  # ⑥ 값
print(f'    ⑦ 실현 변동성:        {rv_val:.4f}  (최근 20일, 연율화)')  # ⑦ 값
print('=' * 60)  # 구분선

## Q14. 과거 월별 국면 분류 히스토리 + 타임라인 시각화
- `.pct_change()`, `scaler.transform()` 등을 채우세요

In [ ]:
# ── Cell 15-A: 과거 월별 국면 분류 테이블 ─────────────────────────────────
from matplotlib.patches import Patch  # 범례 패치 임포트

# ── 월별 국면 테이블 ──
history = features[FEATURE_NAMES + ['phase']].copy()  # 피처 + 국면 복사
history['국면'] = history['phase'].map(PHASE_NAMES)  # 국면 이름 매핑
history.index.name = '날짜'  # 인덱스 이름 설정

# 연도별 그룹으로 출력
print('과거 월별 국면 분류')  # 제목 출력
print('=' * 90)  # 구분선
for year in sorted(history.index.year.unique()):  # 연도별 순회
    rows = history[history.index.year == year]  # 해당 연도 데이터
    print(f'\n📅 {year}년')  # 연도 출력
    print(f'  {"월":>4s}  {"국면":20s}  {"펀더멘털괴리":>10s}  {"ERP Z":>8s}  {"잔차상관":>8s}  {"HY OAS":>8s}  {"실현Vol":>8s}')  # 헤더
    print(f'  {"─"*4}  {"─"*20}  {"─"*10}  {"─"*8}  {"─"*8}  {"─"*8}  {"─"*8}')  # 구분선
    for dt, row in rows.iterrows():  # 각 행에 대해
        print(f'  {dt.month:4d}  {row["국면"]:20s}  {row["fundamental_gap"]:+10.4f}  {row["erp_zscore"]:8.4f}  {row["residual_corr"]:8.4f}  {row["hy_spread"]:8.2f}  {row["realized_vol"]:8.3f}')  # 값 출력

In [ ]:
# ── Cell 15-B: 타임라인 시각화 ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [2, 1, 1]})  # 3행 서브플롯

# [상] S&P 500 + 국면 배경색
sp_sorted = shiller['P'].sort_index()  # S&P 500 가격 정렬
sp_aligned = sp_sorted.reindex(history.index, method='ffill')  # 피처 인덱스에 맞춤

axes[0].plot(history.index, sp_aligned, color='black', linewidth=1, alpha=0.8)  # 가격 플롯
prev_phase = None  # 이전 국면 초기화
span_start = history.index[0]  # 구간 시작
for i, (dt, row) in enumerate(history.iterrows()):  # 각 행에 대해
    ph = row['phase']  # 현재 국면
    if ph != prev_phase and prev_phase is not None:  # 국면이 변하면
        axes[0].axvspan(span_start, dt, alpha=0.2, color=PHASE_COLORS[prev_phase])  # 배경색
        span_start = dt  # 새 구간 시작
    prev_phase = ph  # 이전 국면 갱신
axes[0].axvspan(span_start, history.index[-1], alpha=0.2, color=PHASE_COLORS[prev_phase])  # 마지막 구간
axes[0].set_title('S&P 500 + 국면 타임라인')  # 제목
axes[0].set_ylabel('S&P 500')  # Y축 라벨
axes[0].grid(alpha=0.3)  # 격자
legend_elements = [Patch(facecolor=PHASE_COLORS[k], alpha=0.4, label=PHASE_NAMES[k]) for k in range(4)]  # 범례
axes[0].legend(handles=legend_elements, loc='upper left', fontsize=9)  # 범례 추가

# [중] 국면 스트립
for i, (dt, row) in enumerate(history.iterrows()):  # 각 행에 대해
    axes[1].axvspan(dt, dt + pd.DateOffset(months=1), color=PHASE_COLORS[row['phase']], alpha=0.8)  # 색상 바
axes[1].set_yticks([])  # y축 눈금 제거
axes[1].set_title('월별 국면 (색상 바)')  # 제목
axes[1].set_xlim(history.index[0], history.index[-1] + pd.DateOffset(months=1))  # x축 범위

# [하] noise_score 추정
monthly_noise = []  # 월별 noise_score 리스트
for dt, row in history.iterrows():  # 각 행에 대해
    vals = row[FEATURE_NAMES].values.astype(float)  # 피처 값 추출
    scaled = scaler.___(vals.reshape(1, -1))  # 스케일링 변환 (Q14-1)
    ns = compute_noise_score(scaled)[0]  # noise_score 계산
    monthly_noise.append(ns)  # 결과 추가

history['noise_score'] = monthly_noise  # noise_score 컬럼 추가
axes[2].bar(history.index, history['noise_score'],
            width=25, color=[PHASE_COLORS[p] for p in history['phase']], alpha=0.8)  # 바 차트
axes[2].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[2].set_title('월별 Noise Score v2 (높을수록 센티멘트 지배)')  # 제목
axes[2].set_ylabel('noise_score')  # Y축 라벨
axes[2].grid(alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

In [ ]:
# ── Cell 15-C: 국면 전환 이벤트 ──────────────────────────────────────────
print('\n\n국면 전환 이벤트')  # 전환 이벤트 제목
print('=' * 60)  # 구분선
prev = None  # 이전 국면 초기화
for dt, row in history.iterrows():  # 각 행에 대해
    ph = row['phase']  # 현재 국면
    if ph != prev and prev is not None:  # 국면이 변하면
        print(f'  {dt.strftime("%Y-%m")}  {PHASE_NAMES[prev]} → {PHASE_NAMES[ph]}')  # 전환 출력
    prev = ph  # 이전 국면 갱신
print(f'\n총 {len(history)}개월, 국면 전환 {sum(1 for i in range(1, len(history)) if history["phase"].iloc[i] != history["phase"].iloc[i-1])}회')  # 요약

## Q15. noise_score v2 진단 — 상태별 변수 기여도 분해
- `np.abs()`, `np.zeros()`, `np.column_stack()` 등을 채우세요
- `.pct_change()`, `.corr()` 등을 채우세요

In [ ]:
# ── Cell 16-A: noise_score 변수 기여도 분해 ──────────────────────────────
from matplotlib.patches import Patch  # 범례 패치 임포트

contrib_labels = ['|fund_gap|×0.5', '|erp_z|×0.3', 'resid_corr×1.0',
                  'disp×0', 'amihud×0.5', 'vix_term×2.0', 'hy_sprd×1.5', 'real_vol×2.0']  # 기여도 라벨

def decompose_noise_v2(means_scaled):  # noise_score 변수별 기여도 분해 함수
    """각 변수의 noise_score v2 기여분."""  # 독스트링
    return np.column_stack([  # 열 방향으로 스택
        ___ * np.___(means_scaled[:, 0]),  # fund_gap 절대값 × 가중치 (Q15-1, Q15-2)
        ___ * np.___(means_scaled[:, 1]),  # erp_z 절대값 × 가중치 (Q15-3, Q15-4)
        ___ * means_scaled[:, 2],  # resid_corr × 가중치 (Q15-5)
        np.___(len(means_scaled)),  # dispersion 제거 (0으로 채움) (Q15-6)
        ___ * means_scaled[:, 4],  # amihud × 가중치 (Q15-7)
        ___ * means_scaled[:, 5],  # vix_term × 가중치 (Q15-8)
        ___ * means_scaled[:, 6],  # hy_spread × 가중치 (Q15-9)
        ___ * means_scaled[:, 7],  # realized_vol × 가중치 (Q15-10)
    ])

contrib = decompose_noise_v2(model.means_)  # 기여도 분해 실행
means_orig = scaler.___(model.means_)  # 원본 스케일로 역변환 (Q15-11)

print('(A) noise_score v2 변수 기여도 분해 (scaled × 가중치)')  # 제목
print('=' * 110)  # 구분선
header = f'{"상태":>8s}  {"국면":20s}  {"총점":>6s}  ' + '  '.join(f'{l:>14s}' for l in contrib_labels)  # 헤더
print(header)  # 헤더 출력
print('-' * len(header))  # 구분선

for sid in range(N_STATES):  # 각 상태에 대해
    ph = state_to_phase[sid]  # 국면 매핑
    ns = noise_scores[sid]  # noise_score
    c = contrib[sid]  # 기여도 값
    row = f'state {sid:d}  {PHASE_NAMES[ph]:20s}  {ns:6.2f}  '  # 행 시작
    row += '  '.join(f'{v:14.3f}' for v in c)  # 기여도 값 추가
    print(row)  # 결과 출력

In [ ]:
# ── Cell 16-B: 상태별 시기 + S&P 500 수익률 대조 ─────────────────────────
print('\n(B) 상태별 해당 시기')  # 제목
print('=' * 70)  # 구분선
for sid in range(N_STATES):  # 각 상태에 대해
    ph = state_to_phase[sid]  # 국면 매핑
    mask = features['state'] == sid  # 상태 마스크
    dates = features.index[mask]  # 해당 날짜
    year_counts = pd.Series(dates.year).value_counts().sort_index()  # 연도별 카운트
    periods = [f'{y}({cnt})' for y, cnt in year_counts.items()]  # 연도(수) 형식
    print(f'  state {sid} → {PHASE_NAMES[ph]}')  # 상태 → 국면 출력
    print(f'    기간: {", ".join(periods)}  |  총 {len(dates)}개월')  # 기간 출력

print('\n\n(C) 상태별 S&P 500 월간 수익률')  # 제목
print('=' * 70)  # 구분선
sp_ret = shiller['P'].sort_index().___()  # S&P 500 월간 수익률 (Q15-12)
sp_ret_aligned = sp_ret.reindex(features.index)  # 피처 인덱스에 맞춤

for sid in range(N_STATES):  # 각 상태에 대해
    ph = state_to_phase[sid]  # 국면 매핑
    mask = features['state'] == sid  # 상태 마스크
    rets = sp_ret_aligned[mask].___()  # 결측 제거 (Q15-13)
    pos_pct = (rets > 0).___() * 100  # 상승 비율 (Q15-14)
    print(f'  state {sid} → {PHASE_NAMES[ph]}:')  # 상태 → 국면 출력
    print(f'    평균: {rets.___() *100:+.2f}%  중위수: {rets.median()*100:+.2f}%  '  # 평균, 중위수 (Q15-15)
          f'최대: {rets.max()*100:+.1f}%  최소: {rets.min()*100:+.1f}%  상승월: {pos_pct:.0f}%')  # 최대, 최소, 상승비율

In [ ]:
# ── Cell 16-C: 진단 시각화 (4패널) ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))  # 2×2 서브플롯

# [좌상] 변수별 기여도 스택바
x = np.arange(N_STATES)  # x축 위치
state_labels = [f'state {sid}\n{PHASE_NAMES[state_to_phase[sid]]}' for sid in range(N_STATES)]  # 상태 라벨
colors_contrib = ['#2196F3', '#FF9800', '#9C27B0', '#BDBDBD', '#E91E63', '#FF5722', '#795548', '#3F51B5']  # 색상
bottom_pos = np.zeros(N_STATES)  # 양수 바닥값
bottom_neg = np.zeros(N_STATES)  # 음수 바닥값

for j in range(8):  # 8개 변수에 대해
    vals = contrib[:, j]  # 기여도 값
    pos_vals = np.where(vals >= 0, vals, 0)  # 양수만 추출
    neg_vals = np.where(vals < 0, vals, 0)  # 음수만 추출
    axes[0, 0].bar(x, pos_vals, bottom=bottom_pos, color=colors_contrib[j],
                   alpha=0.8, label=contrib_labels[j])  # 양수 스택바
    axes[0, 0].bar(x, neg_vals, bottom=bottom_neg, color=colors_contrib[j], alpha=0.8)  # 음수 스택바
    bottom_pos += pos_vals  # 양수 바닥값 갱신
    bottom_neg += neg_vals  # 음수 바닥값 갱신

axes[0, 0].set_xticks(x)  # x축 눈금
axes[0, 0].set_xticklabels(state_labels, fontsize=8)  # x축 라벨
axes[0, 0].set_title('noise_score v2 변수별 기여도 분해')  # 제목
axes[0, 0].set_ylabel('기여도 (scaled × weight)')  # Y축 라벨
axes[0, 0].legend(fontsize=6, loc='upper left')  # 범례
axes[0, 0].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[0, 0].grid(axis='y', alpha=0.3)  # 격자
for i in range(N_STATES):  # 총점 표시
    axes[0, 0].text(i, noise_scores[i] + 0.1, f'{noise_scores[i]:.1f}',
                    ha='center', fontsize=10, fontweight='bold')  # 총점

# [우상] 상태별 원본 변수 평균 히트맵
orig_labels = ['fund_gap', 'erp_z', 'resid_corr', 'disp', 'amihud', 'vix_term', 'hy_sprd', 'real_vol']  # 라벨
im = axes[0, 1].imshow(means_orig, cmap='RdYlGn_r', aspect='auto')  # 히트맵
axes[0, 1].set_yticks(range(N_STATES))  # y축 눈금
axes[0, 1].set_yticklabels(state_labels, fontsize=8)  # y축 라벨
axes[0, 1].set_xticks(range(8))  # x축 눈금
axes[0, 1].set_xticklabels(orig_labels, fontsize=7, rotation=30)  # x축 라벨
axes[0, 1].set_title('상태별 변수 평균 (원본 스케일)')  # 제목
for i in range(N_STATES):  # 각 셀에
    for j in range(8):  # 값 표시
        axes[0, 1].text(j, i, f'{means_orig[i, j]:.3f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=axes[0, 1], shrink=0.8)  # 컬러바

# [좌하] 상태별 S&P 500 수익률 박스플롯
bp_data = []  # 박스플롯 데이터
bp_labels = []  # 박스플롯 라벨
bp_colors = []  # 박스플롯 색상
for sid in range(N_STATES):  # 각 상태에 대해
    ph = state_to_phase[sid]  # 국면
    mask = features['state'] == sid  # 마스크
    rets = sp_ret_aligned[mask].dropna() * 100  # 수익률 (%)
    bp_data.append(rets.values)  # 데이터 추가
    bp_labels.append(f'state {sid}\n{PHASE_NAMES[ph]}')  # 라벨 추가
    bp_colors.append(PHASE_COLORS[ph])  # 색상 추가

bplot = axes[1, 0].boxplot(bp_data, labels=bp_labels, patch_artist=True, showfliers=True)  # 박스플롯
for patch, color in zip(bplot['boxes'], bp_colors):  # 각 박스에
    patch.set_facecolor(color)  # 색상 설정
    patch.set_alpha(0.7)  # 투명도 설정
axes[1, 0].axhline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[1, 0].set_title('상태별 S&P 500 월간 수익률 분포 (%)')  # 제목
axes[1, 0].set_ylabel('월간 수익률 (%)')  # Y축 라벨
axes[1, 0].grid(axis='y', alpha=0.3)  # 격자

# [우하] 변수 vs 수익률 상관 + 가중치 비교
corr_vals = []  # 상관 값 리스트
weight_vals = [0.5, 0.3, 1.0, 0.0, 0.5, 2.0, 1.5, 2.0]  # 가중치 리스트
for fname in FEATURE_NAMES:  # 각 피처에 대해
    corr_vals.append(features[fname].___(sp_ret_aligned))  # 수익률과의 상관 (Q15-16)

y = np.arange(8)  # y축 위치
axes[1, 1].barh(y - 0.15, corr_vals, height=0.3, color='#F44336', alpha=0.7, label='corr vs S&P ret')  # 상관 바
axes[1, 1].barh(y + 0.15, [-w/10 for w in weight_vals], height=0.3, color='#2196F3', alpha=0.7, label='noise weight (×-0.1)')  # 가중치 바
axes[1, 1].set_yticks(y)  # y축 눈금
axes[1, 1].set_yticklabels(FEATURE_NAMES, fontsize=8)  # y축 라벨
axes[1, 1].axvline(0, color='gray', linewidth=0.5)  # 0 기준선
axes[1, 1].set_title('변수 상관 vs noise 가중치 (방향 일치 확인)')  # 제목
axes[1, 1].legend(fontsize=8)  # 범례
axes[1, 1].grid(axis='x', alpha=0.3)  # 격자

plt.tight_layout()  # 레이아웃 정리
plt.show()  # 그래프 출력

In [ ]:
# ── Cell 16-D: v1 vs v2 비교 요약 ────────────────────────────────────────
print('\n\n(E) noise_score v1 → v2 변경 요약')  # 제목 출력
print('=' * 70)  # 구분선
print('''
v1 문제점 → v2 해결:
  1. |fund_gap| 가중치 1.0 → 0.5 (corr=+0.11, 수익률과 양의 상관 → 축소)
  2. |erp_zscore| 가중치 1.0 → 0.3 (corr≈0, 거의 무관 → 대폭 축소)
  3. -dispersion 가중치 -1.0 → 0 (방향 왜곡 유발 → 제거)
  4. vix_term 가중치 1.0 → 2.0 (corr=-0.22, 가장 강한 상관 → 상향)
  5. hy_spread 가중치 1.0 → 1.5 (corr=-0.13 → 상향)
  6. realized_vol 신규 추가 (가중치 2.0, 변동성 = 가장 직접적 noise 지표)
''')  # v1 → v2 변경 내용 출력

---
# 정답표 (Answer Key)

| 번호 | 빈칸 | 정답 |
|------|------|------|
| Q1-1 | `from hmmlearn.hmm import ___` | `GaussianHMM` |
| Q1-2 | `from sklearn.preprocessing import ___` | `RobustScaler` |
| Q1-3 | `N_STATES = ___` | `4` |
| Q1-4 | `FEATURE_NAMES = ['___', '___', '___',` | `'fundamental_gap', 'erp_zscore', 'residual_corr'` |
| Q1-5 | `'___', '___', '___', '___',` | `'dispersion', 'amihud', 'vix_term', 'hy_spread'` |
| Q1-6 | `'___']` | `'realized_vol'` |
| Q2-1 | `min(___, month)` | `12` |
| Q2-2 | `shiller.___(subset=['date'])` | `dropna` |
| Q2-3 | `shiller['E'].___()` | `ffill` |
| Q2-4 | `shiller['E'].___(___, min_periods=60)` | `rolling`, `120` |
| Q2-5 | `.___()` | `mean` |
| Q2-6 | `___(shiller['P'])` | `np.log` |
| Q2-7 | `___(shiller['E'].___(...))` | `np.log` |
| Q2-8 | `.___( lower=0.01)` | `clip` |
| Q2-9 | `shiller['log_P'].___(___) ` | `diff`, `12` |
| Q2-10 | `shiller['log_E'].___(___) ` | `diff`, `12` |
| Q2-11 | `fundamental_gap.___()` | `dropna` |
| Q2-12 | `fundamental_gap.___()` | `mean` |
| Q2-13 | `fundamental_gap.___()` | `std` |
| Q3-1~4 | `.___()` (4곳 동일) | `dropna` |
| Q4-1 | `shiller['CAPE'].___()` | `dropna` |
| Q4-2 | `shiller['E'].___(120, ...)` | `rolling` |
| Q4-3 | `.___()` | `mean` |
| Q4-4 | `cape_series.___()` | `dropna` |
| Q4-5 | `}).___()` | `dropna` |
| Q4-6 | `erp.___(___, min_periods=60)` | `rolling`, `120` |
| Q4-7 | `.___()` | `mean` |
| Q4-8 | `erp.___(___, min_periods=60)` | `rolling`, `120` |
| Q4-9 | `.___()` | `std` |
| Q4-10 | `).___()` | `abs` |
| Q4-11 | `erp_zscore.___()` | `dropna` |
| Q4-12 | `erp_zscore.___()` | `mean` |
| Q4-13 | `erp_zscore.___()` | `std` |
| Q5-1 | `stock_prices.___()` | `pct_change` |
| Q5-2 | `.___()` | `dropna` |
| Q5-3 | `ret.___(60, min_periods=30)` | `rolling` |
| Q5-4 | `.___(spy_ret)` | `cov` |
| Q5-5 | `spy_ret.___(60, min_periods=30)` | `rolling` |
| Q5-6 | `.___()` | `var` |
| Q5-7 | `resid_df[s1].___(window)` | `rolling` |
| Q5-8 | `.___(resid_df[s2])` | `corr` |
| Q5-9 | `.___(___)` (쌍별 평균) | `mean`, `axis=1` |
| Q5-10 | `.___(___)` (섹터 평균) | `mean`, `axis=1` |
| Q5-11 | `.___()` | `dropna` |
| Q5-12 | `.___()` (월별) | `mean` |
| Q5-13 | `.___()` | `dropna` |
| Q5-14 | `.___(axis=1)` | `std` |
| Q5-15 | `.___(20)` | `rolling` |
| Q5-16 | `.___()` | `mean` |
| Q5-17 | `.___()` | `dropna` |
| Q5-18 | `.___()` (월별) | `mean` |
| Q5-19 | `.___()` | `dropna` |
| Q6-1 | `___(df_t['Close']/df_t['Open'])` | `np.log` |
| Q6-2 | `.___()` | `abs` |
| Q6-3 | `.___(___) ` (종목 평균) | `mean`, `axis=1` |
| Q6-4 | `.___(___) ` | `dropna` |
| Q6-5 | `.___(20)` | `rolling` |
| Q6-6 | `.___()` | `mean` |
| Q6-7 | `.___()` | `dropna` |
| Q6-8 | `___(amihud_rolling + 1e-15)` | `np.log` |
| Q6-9 | `.quantile(___)` | `0.01` |
| Q6-10 | `.quantile(___)` | `0.99` |
| Q6-11 | `log_amihud.___(q01, q99)` | `clip` |
| Q6-12 | `.___()` (월별) | `mean` |
| Q6-13 | `.___()` | `dropna` |
| Q7-1 | `}).___()` | `dropna` |
| Q7-2 | `vix_term.___()` | `dropna` |
| Q7-3 | `vix_term.___()` | `mean` |
| Q7-4 | `vix_term.___()` | `std` |
| Q7-5 | `(vix_term > 1.0).___()` | `mean` |
| Q7-6 | `hy_spread.___()` | `mean` |
| Q7-7 | `hy_spread.___()` | `std` |
| Q8-1 | `spy_ret_clean.___(20)` | `rolling` |
| Q8-2 | `.___()` | `std` |
| Q8-3 | `___(___) ` (연율화) | `np.sqrt` |
| Q8-4 | 연율화 일수 | `252` |
| Q8-5 | `.___()` (월별) | `mean` |
| Q8-6 | `.___()` | `dropna` |
| Q8-7 | `s.___()` | `dropna` |
| Q8-8 | `features.___()` | `dropna` |
| Q8-9 | `.quantile(___)` | `0.01` |
| Q8-10 | `.quantile(___)` | `0.99` |
| Q8-11 | `features[col].___(q01, q99)` | `clip` |
| Q8-12 | `features.___()` | `corr` |
| Q9-1 | `___()` (스케일러) | `RobustScaler` |
| Q9-2 | `scaler.___(X)` | `fit_transform` |
| Q9-3 | 첫번째 공분산 타입 | `full` |
| Q9-4 | 두번째 공분산 타입 | `diag` |
| Q9-5 | `___(...)` (HMM 모델) | `GaussianHMM` |
| Q9-6 | `n_components=___` | `N_STATES` |
| Q9-7 | `n_iter=___` | `200` |
| Q9-8 | `model.___(X_scaled)` | `fit` |
| Q9-9 | `model.___(X_scaled)` | `predict` |
| Q9-10 | `model.___(X_scaled)` | `score` |
| Q10-1 | `___ * np.abs(...)` (fund_gap 가중치) | `0.5` |
| Q10-2 | `np.___(means[:, 0])` | `abs` |
| Q10-3 | `___ * np.abs(...)` (erp_z 가중치) | `0.3` |
| Q10-4 | `np.___(means[:, 1])` | `abs` |
| Q10-5 | `___ * means[:, 2]` (resid_corr) | `1.0` |
| Q10-6 | `___ * means[:, 4]` (amihud) | `0.5` |
| Q10-7 | `___ * means[:, 5]` (vix_term) | `2.0` |
| Q10-8 | `___ * means[:, 6]` (hy_spread) | `1.5` |
| Q10-9 | `___ * means[:, 7]` (realized_vol) | `2.0` |
| Q10-10 | `np.___(noise_scores)` | `argsort` |
| Q12-1 | `scaler.___(...)` | `inverse_transform` |
| Q13-1 | `recent_resid[s1].___(recent_resid[s2])` | `corr` |
| Q13-2 | `np.___(pair_corrs)` | `mean` |
| Q13-3 | `recent_ret.___(axis=1)` | `std` |
| Q13-4 | `.___()` | `mean` |
| Q13-5 | `___(recent_t['Close']/recent_t['Open'])` | `np.log` |
| Q13-6 | `.___()` | `abs` |
| Q13-7 | `.___(axis=1)` | `mean` |
| Q13-8 | `.___()` | `dropna` |
| Q13-9 | `___(amihud_recent.___() + 1e-15)` | `np.log` |
| Q13-10 | `.___()` (평균) | `mean` |
| Q13-11 | `np.___(log_ami, q01, q99)` | `clip` |
| Q13-12 | `df_vix['vix'].___()` | `dropna` |
| Q13-13 | `df_vix3m['vix3m'].___()` | `dropna` |
| Q13-14 | `df_hy['hy_spread'].___()` | `dropna` |
| Q13-15 | `spy_ret_recent.___()` | `std` |
| Q13-16 | `___(___) ` (연율화) | `np.sqrt` |
| Q13-17 | 연율화 일수 | `252` |
| Q13-18 | `scaler.___(daily_features)` | `transform` |
| Q13-19 | `model.___(daily_scaled)` | `predict_proba` |
| Q14-1 | `scaler.___(vals.reshape(1, -1))` | `transform` |
| Q15-1 | `___ * np.abs(...)` (fund_gap) | `0.5` |
| Q15-2 | `np.___(...)` | `abs` |
| Q15-3 | `___ * np.abs(...)` (erp_z) | `0.3` |
| Q15-4 | `np.___(...)` | `abs` |
| Q15-5 | `___ * means[:, 2]` | `1.0` |
| Q15-6 | `np.___(len(...))` | `zeros` |
| Q15-7 | `___ * means[:, 4]` | `0.5` |
| Q15-8 | `___ * means[:, 5]` | `2.0` |
| Q15-9 | `___ * means[:, 6]` | `1.5` |
| Q15-10 | `___ * means[:, 7]` | `2.0` |
| Q15-11 | `scaler.___(model.means_)` | `inverse_transform` |
| Q15-12 | `shiller['P'].sort_index().___()` | `pct_change` |
| Q15-13 | `sp_ret_aligned[mask].___()` | `dropna` |
| Q15-14 | `(rets > 0).___()` | `mean` |
| Q15-15 | `rets.___()` | `mean` |
| Q15-16 | `features[fname].___(sp_ret_aligned)` | `corr` |